# Week 5 Semantic Search Evaluation

This notebook reviews embedding-based listing search on a fixed 10k sample, then compares MiniLM, MPNet, and BM25 keyword retrieval.


In [1]:
import sys
import time

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", None)

sys.path.append("..")

from src.real_estate_nlp.keyword_search import BM25Searcher
from src.real_estate_nlp.semantic_search import SemanticSearcher

---

## 1. Artifacts Loading

The 10k sample is the working set for latency checks and retrieval review.

In [2]:
sample = pd.read_csv("../data/processed/listing_semantic_sample_10k.csv")
sample.shape

(10000, 9)

In [3]:
sample[["listing_id", "city", "price", "beds", "baths", "sqft", "remarks_cleaned"]].head(3)

,listing_id,city,price,beds,baths,sqft,remarks_cleaned
0,1149776480,Irvine,899000,2.0,2.0,1260.0,"luxury top floor penthouse life style with panoramic pool views at watermarke, irvine. welcome this top floor 2 bedroom, 2 bath condo offers total privacy with no neighbors above, no neighbor next door, and unobstructed views of the resort style pool, this residence blends sophistication, comfort, and exclusivity. the home welcomes you with a bright, expansive open layout and an upgraded gourmet kitchen appointed with granite countertops, stainless steel appliances, and elegant custom cabinetry. the primary suite provides a tranquil retreat with serene pool vistas, a spacious walk in closet, and a luxuriously appointed en suite bath. the secondary bedroom is equally well designed, featuring generous space and storage. a noteworthy highlight of this 1260 square feet of spacious living property is the inclusion of 2 private, covered parking spaces located on the same level, 4th floor a rare and highly desirable convenience within the community. an in unit laundry room adds to the overall ease and functionality of the home. residents enjoy access to watermarke's extensive amenity collection, including a 7 days a week concierge, a grand clubhouse with movie theater and business center, a junior olympic sized heated pool, private gated area, multiple spas, a state of the art fitness center, tennis and basketball courts, and beautifully landscaped grounds. situated moments from uc irvine, john wayne airport, irvine spectrum, premier shopping, dining, and irvine's finest conveniences, this penthouse offers an exceptional blend of luxury and accessibility."
1,1150462921,Menifee,630000,4.0,2.0,1982.0,"welcome home to stunning views and 1 story living in menifee enjoy breathtaking valley views and unforgettable sunsets from this highly upgraded 4 bedroom, 2 bath, 1 story home located in the heart of menifee. inside, you'll find an open concept floor plan featuring upgraded flooring, elegant arched doorways, vaulted ceilings, and floor to ceiling windows that flood the home with natural light while showcasing the spectacular views. the formal living room flows effortlessly into the main living space, where the kitchen shines with a raised center island breakfast bar, gleaming tile countertops, ample cabinetry, and pull out drawers for added functionality.the kitchen opens to the dining area and family room, complete with a cozy fireplace and direct access to the backyard perfect for entertaining. the spacious primary suite offers private backyard access and a luxurious ensuite with dual sink vanities, a separate soaking tub and shower, and a large walk in closet. three additional bedrooms are generously sized, ideal for family, guests, or a home office.step outside to your private oasis featuring lush, mature landscaping, rock accents, a cement retaining wall, drip irrigation system, and vinyl fencing. relax under the expansive covered patio with built in lighting and an automatic electric shade, the perfect spot to unwind while enjoying menifee's stunning sunsets.additional highlights include plantation shutters throughout, art accent lighting, inside laundry, a 3 car garage, this is a rare opportunity to own a beautifully upgraded home with incredible views hurry, this one won't last"
2,1168736420,Long Barn,524900,3.0,2.0,1985.0,"what a fun place to live full time or go spend quality time . this is a move in ready mountain retreat in the special gated sierra park community a short drice to pinecrest lake and dodge ridge ski area. situated on a desirable double lot, this custom 1985 square feet home offers 3 bedroom, 2 bathroom, a deep oversized 2 car garage, carport, and composition roof. just 15 miles from pinecrest lake and dodge ridge ski resort, with sierra park amenities including a private swimming lake, playground, and lodge. the spacious great room features a high coffered ceiling, cozy fireplace, skylights, sun tubes, and abundant

---

## 2. Search Indexes

The two semantic searchers load saved FAISS indexes built from the same 10k records. BM25 uses the same text as a keyword baseline.

In [4]:
records = sample.to_dict("records")

minilm = SemanticSearcher(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    local_files_only=True,
    batch_size=8,
).load("../data/models/semantic/sentence-transformers_all-MiniLM-L6-v2", "sample_10k")

mpnet = SemanticSearcher(
    model_name="sentence-transformers/all-mpnet-base-v2",
    local_files_only=True,
    batch_size=4,
).load("../data/models/semantic/sentence-transformers_all-mpnet-base-v2", "sample_10k")

bm25 = BM25Searcher().build(records)

embedding_searchers = {"MiniLM": minilm, "MPNet": mpnet}
search_methods = {"MiniLM": minilm, "MPNet": mpnet, "BM25": bm25}

pd.DataFrame([
    {"method": name, "listings": len(searcher.metadata), "embedding_dim": searcher.embeddings.shape[1]}
    for name, searcher in embedding_searchers.items()
])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,method,listings,embedding_dim
0,MiniLM,10000,384
1,MPNet,10000,768


Two models are used and compared in this study.

- `all-MiniLM-L6-v2` (default model)
  - small and fast
  - 384-dimensional
- `all-mpnet-base-v2`
  - heavy
  - 768-dimensional

---

## 3. Retrieval Examples

We first check whether the top results read like a reasonable answer to the query.

In [5]:
query = "home with a private pool and outdoor entertaining space"

def compact_results(results):
    cols = ["rank", "score", "listing_id", "city", "price", "beds", "baths", "text"]
    df = pd.DataFrame(results)
    df["text"] = df["search_text"]
    return df[cols]

example_results = []
for method, searcher in search_methods.items():
    result = compact_results(searcher.search(query, top_k=3))
    result.insert(0, "method", method)
    example_results.append(result)

pd.concat(example_results, ignore_index=True)

,method,rank,score,listing_id,city,price,beds,baths,text
0,MiniLM,1,0.737546,1111896555,Fontana,625000,4.0,2.0,"large 4 bedroom 2 bathroom home with living room dining off kitchen. enclosed patio 300 square feet pool. close to schools, shopping freeways."
1,MiniLM,2,0.733873,1159158818,Blythe,245000,3.0,2.0,"pool, pool home this can be your vacation home, investmen o short tem rental airbnb. this inviting 3 bedroom, 2 bathroom 1 story home sits in a quiet neighborhood just minutes from the scenic colorado river, offering a perfect blend of comfort and outdoor lifestyle. step outside to a spacious backyard oasis, highlighted by a private swimming pool perfect for cooling off during hot summer days. the yard also include a covered patio area, ideal for barbecues, relaxing evenings, or enjoying views of the surrounding landscape. the home features an open concept living area with plenty of natural light, connecting the living room, dining space, and a good size kitchen. the layout is designed for both everyday living and entertaining. stay comfortable year round with central air conditioning system. the primary bedroom includes a private en suite bathroom and generous closet space, while the two additional bedrooms are ideal for family, guests, or a home office. a second full bathroom is conveniently located nearby. both bathrooms have been upgrated.this home also offers fresh interior exterior paint, new carpet. few minutes away from 10fwy"
2,MiniLM,3,0.696034,1158583082,Palmdale,479900,4.0,2.0,"ready for your high efficiency pool home here it is palmdale pool home with 4 bedroom and 2 bathroom. this home features triple pane windows, new tankless water heater, upgraded electrical panel, foam insulated walls, and zone heating and cooling. 2 evaporative coolers and central heat, plus gas stove. whole home water filtration system. living room has vaulted wood ceilings with large windows that create a bright open space. remodeled kitchen with new stainless steel appliances. spacious master bedroom with 2 closets 1 walk in . remodeled en suite master bathroom. 3 secondary bedrooms. remodeled hall bath with jacuzzi tub. new carpet, faux wood window covers, and paint throughout. the private backyard is a great entertaining space with 2 covered patio areas. enjoy the pebble tech in ground pool, with all upgraded plumbing and newer equipment. fully fenced for safety."
3,MPNet,1,0.763622,1159158818,Blythe,245000,3.0,2.0,"pool, pool home this can be your vacation home, investmen o short tem rental airbnb. this inviting 3 bedroom, 2 bathroom 1 story home sits in a quiet neighborhood just minutes from the scenic colorado river, offering a perfect blend of comfort and outdoor lifestyle. step outside to a spacious backyard oasis, highlighted by a private swimming pool perfect for cooling off during hot summer days. the yard also include a covered patio area, ideal for barbecues, relaxing evenings, or enjoying views of the surrounding landscape. the home features an open concept living area with plenty of natural light, connecting the living room, dining space, and a good size kitchen. the layout is designed for both everyday living and entertaining. stay comfortable year round with central air conditioning system. the primary bedroom includes a private en suite bathroom and generous closet space, while the two additional bedrooms are ideal for family, guests, or a home office. a second full bathroom is conveniently located nearby. both bathrooms have been upgrated.this home also offers fresh interior exterior paint, new carpet. few minutes away from 10fwy"
4,MPNet,2,0.738082,1156477613,Riverside,865000,4.0,4.0,"location location welcome to your private pool spa home. in orangecrest experience the best of southern california living in this beautifully upgraded home, where a resort style backyard and thoughtfully designed interior come together for the ultimate in comfort and entertaining. from the moment you arrive, stunning manicured curb app

---

## 4. Comparison Query Set

The query labels use graded proxy relevance rather than a single exact keyword list. Each query has three clue groups:

- `core`: direct evidence for the requested feature
- `related`: strong nearby language or common real estate variants
- `supporting`: weaker evidence that still points in the right direction

This is still not a human-labeled benchmark, but it is less brittle than exact term matching.

In [6]:
eval_queries = [
    {
        "query": "private pool with an outdoor dining area",
        "core": ["pool", "spa"],
        "related": ["outdoor kitchen", "barbecue", "bbq", "covered patio", "deck", "terrace", "cabana"],
        "supporting": ["patio", "courtyard", "outdoor", "entertain", "backyard"],
    },
    {
        "query": "ocean view home close to the beach",
        "core": ["ocean view", "water view", "coastline view", "catalina view"],
        "related": ["beach", "coastal", "ocean", "shoreline"],
        "supporting": ["walk to beach", "near the beach", "waves", "harbor"],
    },
    {
        "query": "remodeled kitchen with modern appliances",
        "core": ["remodeled kitchen", "updated kitchen", "renovated kitchen", "gourmet kitchen", "chef"],
        "related": ["stainless steel", "quartz", "granite", "kitchen island", "custom cabinetry"],
        "supporting": ["modern", "upgraded", "new appliances", "countertops"],
    },
    {
        "query": "single story home with open living space",
        "core": ["single story", "one story", "single level", "single-story"],
        "related": ["open floor", "open concept", "great room", "open living"],
        "supporting": ["spacious living", "vaulted", "flow", "open layout"],
    },
    {
        "query": "home near parks and schools",
        "core": ["school", "park"],
        "related": ["playground", "trail", "recreation", "walking distance"],
        "supporting": ["nearby", "close to", "community", "neighborhood"],
    },
    {
        "query": "condo with resort style amenities",
        "core": ["condo", "condominium", "resort style", "resort-style"],
        "related": ["clubhouse", "fitness center", "pool", "spa", "tennis", "concierge"],
        "supporting": ["amenities", "gated", "lounge", "community"],
    },
    {
        "query": "large lot with room to build",
        "core": ["large lot", "oversized lot", "acre", "acres", "room to build"],
        "related": ["adu", "guest house", "expand", "expansion", "development"],
        "supporting": ["rv parking", "usable lot", "flat lot", "yard space"],
    },
    {
        "query": "move in ready home with recent upgrades",
        "core": ["move in ready", "move-in ready", "turnkey", "recently remodeled"],
        "related": ["updated", "upgraded", "renovated", "newly remodeled"],
        "supporting": ["new flooring", "fresh paint", "new roof", "new hvac"],
    },
    {
        "query": "dedicated home office or flexible bonus room",
        "core": ["home office", "office", "study", "bonus room", "flex room"],
        "related": ["loft", "work from home", "media room", "library"],
        "supporting": ["extra room", "multipurpose", "optional bedroom", "guest room"],
    },
    {
        "query": "garage parking with extra storage",
        "core": ["garage", "parking"],
        "related": ["storage", "workshop", "cabinets", "built in storage"],
        "supporting": ["driveway", "carport", "covered parking", "rv parking"],
    },
]

pd.DataFrame(eval_queries)

,query,core,related,supporting
0,private pool with an outdoor dining area,"[pool, spa]","[outdoor kitchen, barbecue, bbq, covered patio, deck, terrace, cabana]","[patio, courtyard, outdoor, entertain, backyard]"
1,ocean view home close to the beach,"[ocean view, water view, coastline view, catalina view]","[beach, coastal, ocean, shoreline]","[walk to beach, near the beach, waves, harbor]"
2,remodeled kitchen with modern appliances,"[remodeled kitchen, updated kitchen, renovated kitchen, gourmet kitchen, chef]","[stainless steel, quartz, granite, kitchen island, custom cabinetry]","[modern, upgraded, new appliances, countertops]"
3,single story home with open living space,"[single story, one story, single level, single-story]","[open floor, open concept, great room, open living]","[spacious living, vaulted, flow, open layout]"
4,home near parks and schools,"[school, park]","[playground, trail, recreation, walking distance]","[nearby, close to, community, neighborhood]"
5,condo with resort style amenities,"[condo, condominium, resort style, resort-style]","[clubhouse, fitness center, pool, spa, tennis, concierge]","[amenities, gated, lounge, community]"
6,large lot with room to build,"[large lot, oversized lot, acre, acres, room to build]","[adu, guest house, expand, expansion, development]","[rv parking, usable lot, flat lot, yard space]"
7,move in ready home with recent upgrades,"[move in ready, move-in ready, turnkey, recently remodeled]","[updated, upgraded, renovated, newly remodeled]","[new flooring, fresh paint, new roof, new hvac]"
8,dedicated home office or flexible bonus room,"[home office, office, study, bonus room, flex room]","[loft, work from home, media room, library]","[extra room, multipurpose, optional bedroom, guest room]"
9,garage parking with extra storage,"[garage, parking]","[storage, workshop, cabinets, built in storage]","[driveway, carport, covered parking, rv parking]"


---

## 5. Latency Study

This section measures search latency on the fixed 10k listing sample.

Two timings are useful:

- End-to-end latency: the practical search time, including query embedding and retrieval.
- FAISS-only latency: a diagnostic lookup time, using already-computed query embeddings against the 10k-vector index.

In [7]:
def milliseconds(values):
    return [round(value * 1000, 2) for value in values]


timing_rows = []

for method, searcher in search_methods.items():
    times = []
    for item in eval_queries:
        start = time.perf_counter()
        searcher.search(item["query"], top_k=10)
        times.append(time.perf_counter() - start)

    timing_rows.append({
        "method": f"{method}_end_to_end",
        "avg_ms": np.mean(milliseconds(times)),
        "p95_ms": np.percentile(milliseconds(times), 95),
    })

for method, searcher in embedding_searchers.items():
    query_embeddings = searcher.encode_queries([item["query"] for item in eval_queries])
    times = []
    for query_embedding in query_embeddings:
        start = time.perf_counter()
        searcher.index.search(query_embedding.reshape(1, -1), 10)
        times.append(time.perf_counter() - start)

    timing_rows.append({
        "method": f"{method}_faiss_lookup_only",
        "avg_ms": np.mean(milliseconds(times)),
        "p95_ms": np.percentile(milliseconds(times), 95),
    })

pd.DataFrame(timing_rows)

,method,avg_ms,p95_ms
0,MiniLM_end_to_end,8.128,16.8440
1,MPNet_end_to_end,13.232,21.3440
2,BM25_end_to_end,8.641,11.5500
3,MiniLM_faiss_lookup_only,0.196,0.2110
4,MPNet_faiss_lookup_only,0.379,0.4055


---

## 6. Retrieval Quality Evaluation

This section uses the same 10-query set to compare top-5 retrieval quality across MiniLM, MPNet, and BM25.

The scoring is a graded proxy, not final human judgment:

- 3 = core clue plus related evidence, or multiple core clues
- 2 = one core clue, or multiple related clues
- 1 = one related or supporting clue
- 0 = no proxy evidence

The three metric tables below report Precision@5, NDCG@5, and MRR separately.

In [8]:
def clue_hits(text, clues):
    return sum(clue in text for clue in clues)


def relevance_grade(result, item):
    text = f"{result.get('city', '')} {result.get('search_text', '')}".lower()
    core_hits = clue_hits(text, item["core"])
    related_hits = clue_hits(text, item["related"])
    supporting_hits = clue_hits(text, item["supporting"])

    if core_hits >= 2 or (core_hits >= 1 and related_hits >= 1):
        return 3
    if core_hits >= 1 or related_hits >= 2:
        return 2
    if related_hits >= 1 or supporting_hits >= 1:
        return 1
    return 0


review_rows = []

for item in eval_queries:
    for method, searcher in search_methods.items():
        for result in searcher.search(item["query"], top_k=5):
            review_rows.append({
                "method": method,
                "query": item["query"],
                "rank": result["rank"],
                "listing_id": result["listing_id"],
                "city": result["city"],
                "score": result["score"],
                "remarks": result["search_text"],
                "_grade": relevance_grade(result, item),
            })

review_df = pd.DataFrame(review_rows)
review_df[["method", "query", "rank", "listing_id", "city", "score", "remarks"]].head(10)

,method,query,rank,listing_id,city,score,remarks
0,MiniLM,private pool with an outdoor dining area,1,1159158818,Blythe,0.675928,"pool, pool home this can be your vacation home, investmen o short tem rental airbnb. this inviting 3 bedroom, 2 bathroom 1 story home sits in a quiet neighborhood just minutes from the scenic colorado river, offering a perfect blend of comfort and outdoor lifestyle. step outside to a spacious backyard oasis, highlighted by a private swimming pool perfect for cooling off during hot summer days. the yard also include a covered patio area, ideal for barbecues, relaxing evenings, or enjoying views of the surrounding landscape. the home features an open concept living area with plenty of natural light, connecting the living room, dining space, and a good size kitchen. the layout is designed for both everyday living and entertaining. stay comfortable year round with central air conditioning system. the primary bedroom includes a private en suite bathroom and generous closet space, while the two additional bedrooms are ideal for family, guests, or a home office. a second full bathroom is conveniently located nearby. both bathrooms have been upgrated.this home also offers fresh interior exterior paint, new carpet. few minutes away from 10fwy"
1,MiniLM,private pool with an outdoor dining area,2,1156235742,Indio,0.648095,"looking for a private desert resort compound welcome to a unique lifestyle opportunity in desert river estates. situated on approximately 21780 square feet half acre lot with exceptional privacy, this beautifully positioned 4 bedroom, 4 bath plus den residence offers 3468 square feet of indoor outdoor living designed for relaxation, entertaining and year round desert enjoyment.few homes in desert river estates offer this level of outdoor experience and lifestyle living. designed by a professional pool builder as his personal showpiece, the backyard is the true heart of the property and delivers an experience that would be extraordinarily expensive to recreate today. the custom pool and spa feature waterfalls, fire elements, bubblers, swim in place system, tanning shelves coupled with multiple gathering areas designed to create your own private desert retreat. a dedicated heat pump cooling system keeps the pool comfortable during warmer summer months.the lush grounds, feature mature date palm and fruit trees, raised bed planters for vegetable or herb gardens, a putting green and outdoor kitchen. extensive behind the wall parking with automatics gate access and 220 hookup is suitable for additional vehicles or rv storage, provided it remains screened from street view.the desirable east facing orientation minimizes harsh afternoon sun and creates a more enjoyable shaded outdoor environment throughout much of the year. this home offers exceptional rear privacy and a true resort style atmosphere designed for quiet enjoyment, entertaining and escape. inside, soaring ceilings and expansive windows create a bright, open atmosphere with beautiful pool and backyard views. a spacious great room with fireplace opens to the island kitchen, creating an ideal setting for entertaining family and guests. the thoughtfully designed floor plan offers excellent separation and privacy, with three en suite guest bedrooms on one side of the home. the private primary suite occupies its own wing with direct patio and pool access.ideally located between indio and la quinta and within approximately 1.5 miles of the coachella and stagecoach festival grounds, this home blends comfort and location in a way that works beautifully as a primary residence, desert retreat or seasonal getaway. desert river estates is known for its quiet elegance, beautifully maintained desert landscaping, low homeowners association dues. a rare sense of privacy make this gated community one of the desert's hidden gems."
2,MiniLM,private pool with an outdoor dining area,3,1174462334,Lake Balboa,0.645861,"entertainer's dream with pool in coveted l

In [9]:
precision_table = (
    review_df.assign(_clear_relevant=review_df["_grade"].ge(2))
    .groupby(["method", "query"], as_index=False)
    .agg(precision_at_5=("_clear_relevant", "mean"))
    .groupby("method", as_index=False)
    .agg(precision_at_5=("precision_at_5", "mean"))
)

precision_table

,method,precision_at_5
0,BM25,0.82
1,MPNet,0.84
2,MiniLM,0.82


In [10]:
def dcg(grades):
    grades = np.asarray(grades)
    discounts = np.log2(np.arange(2, len(grades) + 2))
    return np.sum((np.power(2, grades) - 1) / discounts)


ndcg_rows = []

for query, query_df in review_df.groupby("query"):
    pooled_grades = (
        query_df.groupby("listing_id")["_grade"]
        .max()
        .sort_values(ascending=False)
        .head(5)
        .tolist()
    )
    ideal = dcg(pooled_grades)

    for method, method_df in query_df.groupby("method"):
        ranked = method_df.sort_values("rank")["_grade"].tolist()
        score = dcg(ranked) / ideal if ideal else 0
        ndcg_rows.append({"method": method, "query": query, "ndcg_at_5": score})

ndcg_table = (
    pd.DataFrame(ndcg_rows)
    .groupby("method", as_index=False)
    .agg(ndcg_at_5=("ndcg_at_5", "mean"))
)

ndcg_table

,method,ndcg_at_5
0,BM25,0.857744
1,MPNet,0.780715
2,MiniLM,0.730257


In [11]:
mrr_rows = []

for (method, query), query_df in review_df.groupby(["method", "query"]):
    clear_hits = query_df.sort_values("rank").query("_grade >= 2")
    reciprocal_rank = 0 if clear_hits.empty else 1 / clear_hits.iloc[0]["rank"]
    mrr_rows.append({"method": method, "query": query, "reciprocal_rank": reciprocal_rank})

mrr_table = (
    pd.DataFrame(mrr_rows)
    .groupby("method", as_index=False)
    .agg(mrr=("reciprocal_rank", "mean"))
)

mrr_table

,method,mrr
0,BM25,0.9
1,MPNet,0.9
2,MiniLM,0.9
